## A3.1 Support Vector Machines & Multiple Testing

***

**1.** Se importa la librería *pandas* para cargar el archivo con la base de datos a utilizar.

Antes de comenzar con el trabajo, se revisan las dimensiones del datafrme, se imprimen las primeras filas del conjunto de datos y se verifica que no haya huecos en la misma.

Realizando esta verificación, se filtran los datos en dos subconjuntos, uno para las muestras de la clase 2 y otro para las de la clase 4. Son las filas en las que el valor de la columna *y* es igual a 2 o 4. 

Luego se calcula la diferencia absoluta (ya que solo es relevante el tamaño de la diferencia, no si es positiva o negativa) entre las medias de los genes de las dos clases para cada gen (todas excepto *y* la variable de salida).

Se ordena las diferencias de medias de mayor a menor y selecciona los 10 genes con la mayor diferencia.

In [4]:
import pandas as pd 

data=pd.read_csv('A3.1 Khan.csv')
print("Dimensiones:", data.shape)
print(data.head())
print(data.isnull().sum())

clase2=data[data["y"]==2]
clase4=data[data["y"]==4]

dif_prom=(clase2.iloc[:, :-1].mean()-clase4.iloc[:, :-1].mean()).abs()
genes_top=dif_prom.sort_values(ascending=False).head(10)
print("\nTop 10 genes con mayor diferencia de medias entre clase 2 y 4:")
print(genes_top)

Dimensiones: (83, 2309)
         X1        X2        X3        X4        X5        X6        X7  \
0  0.773344 -2.438405 -0.482562 -2.721135 -1.217058  0.827809  1.342604   
1 -0.078178 -2.415754  0.412772 -2.825146 -0.626236  0.054488  1.429498   
2 -0.084469 -1.649739 -0.241308 -2.875286 -0.889405 -0.027474  1.159300   
3  0.965614 -2.380547  0.625297 -1.741256 -0.845366  0.949687  1.093801   
4  0.075664 -1.728785  0.852626  0.272695 -1.841370  0.327936  1.251219   

         X8        X9       X10  ...     X2300     X2301     X2302     X2303  \
0  0.057042  0.133569  0.565427  ... -0.027474 -1.660205  0.588231 -0.463624   
1 -0.120249  0.456792  0.159053  ... -0.246284 -0.836325 -0.571284  0.034788   
2  0.015676  0.191942  0.496585  ...  0.024985 -1.059872 -0.403767 -0.678653   
3  0.819736 -0.284620  0.994732  ...  0.357115 -1.893128  0.255107  0.163309   
4  0.771450  0.030917  0.278313  ...  0.061753 -2.273998 -0.039365  0.368801   

      X2304     X2305     X2306     X2307   

Si ciertos genes tienen una diferencia significativa en su expresión entre las dos clases, esto podría indicar que estos genes están involucrados de manera diferente en los dos tipos de cáncer. Para un estudio de inferencia, una diferencia significativa en las medias podría indicar que los grupos son estadísticamente diferentes en cuanto a la expresión de esos genes. Sin embargo, para ver si estas diferencias son estadísticamente significativas y no solo al azar, se realizará un análisis ANOVA.

***

**2.** Se definen listas vacías *t_stats* y *p_values* esto para almacenar tanto los estadísticos t de la prueba t para cada gen, como los valores p correspondientes a cada uno de los estadísticos t calculados.

Se itera sobre todas las columnas, excepto la última. Dentro de este se realiza una prueba t de student para muestras independientes utilizando las muestras de cada gen para las clases 2 y 4. Esta prueba compara las medias de las dos clases para un gen específico y evalúa si hay una diferencia significativa entre ellas. Al final se coloca que no se asume que las dos clases tienen la misma varianza. Por último se agregan el estadístico t y valor p a las listas correspondientes creadas previamente.

Posterior a esto se aplican los tres métodos: Bonferroni, Holm y Benjamini-Hochberg para corregir por múltiples pruebas. El de Bonferroni y Holm se centran en controlar el error tipo 1, sin embargo Holm es más flexible/menos conservador. Benjamini-Hochberg se centra en controlar la tasa de descubrimientos falsos y permite una mayor tasa de rechazos de la hipóstesis nula.

In [8]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind, f_oneway

t_stats, p_values = [], []
for gen in data.columns[:-1]:
    stat, p=ttest_ind(clase2[gen], clase4[gen], equal_var=False)
    t_stats.append(stat)
    p_values.append(p)

bonferroni=multipletests(p_values, alpha=0.05, method='bonferroni')[0]
holm=multipletests(p_values, alpha=0.05, method='holm')[0]
bh=multipletests(p_values, alpha=0.05, method='fdr_bh')[0]

print("Genes significativos Bonferroni:", list(data.columns[:-1][bonferroni]))
print("\nGenes significativos Holm:", list(data.columns[:-1][holm]))
print("\nGenes significativos Benjamini-Hochberg:", list(data.columns[:-1][bh]))

Genes significativos Bonferroni: ['X2', 'X36', 'X67', 'X129', 'X174', 'X187', 'X188', 'X229', 'X246', 'X251', 'X338', 'X348', 'X368', 'X372', 'X373', 'X380', 'X430', 'X433', 'X509', 'X545', 'X554', 'X558', 'X566', 'X603', 'X655', 'X714', 'X762', 'X910', 'X951', 'X971', 'X1003', 'X1021', 'X1023', 'X1055', 'X1070', 'X1093', 'X1105', 'X1110', 'X1112', 'X1132', 'X1194', 'X1196', 'X1207', 'X1217', 'X1298', 'X1319', 'X1327', 'X1330', 'X1372', 'X1389', 'X1416', 'X1610', 'X1626', 'X1634', 'X1645', 'X1706', 'X1708', 'X1723', 'X1738', 'X1799', 'X1888', 'X1896', 'X1911', 'X1924', 'X1954', 'X1955', 'X1980', 'X2046', 'X2050', 'X2115', 'X2146', 'X2247']

Genes significativos Holm: ['X2', 'X36', 'X67', 'X129', 'X174', 'X187', 'X188', 'X229', 'X246', 'X251', 'X338', 'X348', 'X368', 'X372', 'X373', 'X380', 'X430', 'X433', 'X509', 'X545', 'X554', 'X558', 'X566', 'X603', 'X655', 'X714', 'X762', 'X910', 'X951', 'X971', 'X1003', 'X1021', 'X1023', 'X1055', 'X1070', 'X1093', 'X1105', 'X1110', 'X1112', 'X1132

***

**3.** Se vuelven a crear listas vacías para almacenr los estadísticos f y los valores p de los estadísticos f para cada gen. De igual forma se itera con todas las columnas, excepto la última. Se crea una lista *muestras* para los valores del gen específico en todas las clases. En este caso se utiliza *f_oneway* para realizar la prueba de ANOVA de una vía. En cada iteración se van agregando el estadístico f y el valor p a la lista correspondiente. 

Se aplica la correción con Benjamini-Hochberg ajustando los valores p obtenidos del ANOVA para manejar el problema de las múltiples comparaciones.

Una vez realizado esto, se filtra los genes significativos con base en los valores ajustados del ANOVA.

In [11]:
grupos=[data[data["y"] == c].iloc[:, :-1] for c in sorted(data["y"].unique())]
f_stats, p_vals_anova = [], []

for gen in data.columns[:-1]:
    muestras=[group[gen] for group in grupos]
    f_stat, p_val = f_oneway(*muestras)
    f_stats.append(f_stat)
    p_vals_anova.append(p_val)

anova=multipletests(p_vals_anova, alpha=0.05, method='fdr_bh')
genes_sig_anova=data.columns[:-1][anova[0]]

print("Genes significativos por ANOVA:",list(genes_sig_anova[:10]))

Genes significativos por ANOVA: ['X1', 'X2', 'X3', 'X9', 'X12', 'X17', 'X21', 'X22', 'X27', 'X29']


***

**4.** Se separan las características, donde *X* contiene todas las columnas excepto la objetivo, y en *y* se extrae solo la columna *y* que contiene las etiquetas de clase. Esto es lo que se busca predecir con el modelo.

Se dividen los datos en entrenamiento y prueba con proporción 70/30, así mismo se asegura que la división se haga de manera estratificada, es decir, que las proporciones de entrenamiento y prueba sean similares a las del conjunto original.

Se crean y entrenan modelos SVM con un kernel lineal, polinomial de orden 3 y radial. Durante esto con *StandardScaler* se escalan las caracteristícas (normalizan: media 0 y varianza 1), para posterior aplicar el modelo correspondiente.

Lineal: se espera que las clases sean separables por una línea recta o hiperplano.
Polínomico: captura relaciones no lineales en los datos.
Rdial: mapea los datos a un espacio de características de mayor dimensión, facilitando la separación no lineal.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X=data.drop(columns="y")
y=data["y"]
top_10=genes_top.index
X_top10 = X[top_10]
X_train, X_test, y_train, y_test=train_test_split(X_top10, y, test_size=0.3, random_state=42, stratify=y)

svm_linear=make_pipeline(StandardScaler(), SVC(kernel='linear'))
svm_poly=make_pipeline(StandardScaler(), SVC(kernel='poly', degree=3))
svm_rbf=make_pipeline(StandardScaler(), SVC(kernel='rbf'))

modelos={'lineal': svm_linear,'polinomial grado 3': svm_poly,'radial': svm_rbf}

***

**5.** Se importa la función *classification_report*, esto para generar un reporte con las métricas de precisión, recall, f1-score y accuracy.

Dentro del ciclor for se entrena cada uno de los modelos utilizando los datos de entrenamiento para a partir de estos datos hacer predicciones sobre datos no vistos. Después de entrenar, se utiliza *predict* para realizar predicciones sobre los datos de prueba. Se imprimen los resultados, así como el númeroo de vectores de soporte para cada clase en el conjunto de entrenamiento.

In [17]:
from sklearn.metrics import classification_report

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred=modelo.predict(X_test)
    
    print("\nResultados para SVM con kernel", nombre)
    print(classification_report(y_test, y_pred))
    
    vec_sop = modelo.named_steps['svc'].n_support_
    print("Vectores de soporte modelo",nombre,":", vec_sop)


Resultados para SVM con kernel lineal
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
           2       1.00      1.00      1.00         9
           3       1.00      1.00      1.00         5
           4       1.00      1.00      1.00         8

    accuracy                           1.00        25
   macro avg       1.00      1.00      1.00        25
weighted avg       1.00      1.00      1.00        25

Vectores de soporte modelo lineal : [6 5 7 4]

Resultados para SVM con kernel polinomial grado 3
              precision    recall  f1-score   support

           1       1.00      0.33      0.50         3
           2       1.00      0.89      0.94         9
           3       0.56      1.00      0.71         5
           4       1.00      0.88      0.93         8

    accuracy                           0.84        25
   macro avg       0.89      0.77      0.77        25
weighted avg       0.91      0.84      0.84      

***

**Conclusión:** Tanto el modelo lineal como el radial muestran métricas perfectas, sin embargo esto es esperado debido a la manera en la que se trabajaron los datos. El modelo polinomial es el único que muestra variación dependiendo de las clases. Debido a esto es posible enfocarnos un poco más en los vectores de soporte para evaluar los modelos.

Lineal: la cantidad de vectores de soporte es baja para cada clase, indicando que las clases están bien separadas linealmente en el espacio de características.

Polinomial: este modelo tiene un recall bajo de 0.33 y un f1-score de 0.5, indicando que el modelo batalla para identificar correctamente todos los casos de la clase 1. Así mismo, el número de vectores de soporte es algo alto, mostrando que el modelo está aprendiendo fronteras más complejas para separar las clases.

Radial: la cantidad de vectores de soporte es mayor que en el modelo lineal, por lo que esto puede señalar que el modelo está manejando fronteras de decisión un poco más complejas en este caso.

Debido a que es muy probable que las clases no siempre sean lineales, considero que el más adecuado para este caso es el radial ya que tiene rendimiento perfecto y aunque el número de vectores de soporte es mayor que el modelo lineal, este modelo sigue clasificando correctamente todas las muestras, aprendiendo bien las relaciones entre las clases.